In [1]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from pylab import *
import seaborn as sns
import pdb
from skimage.measure import label, regionprops
from scipy import ndimage as ndi
from scipy.ndimage.measurements import center_of_mass, label
from skimage.feature import peak_local_max
from skimage.segmentation import active_contour,watershed

/var/folders/30/gtfddbxx5z3f_hc8v4br4pkh0000gn/T/ipykernel_74912/1514587937.py:10: DeprecationWarning: Please use `center_of_mass` from the `scipy.ndimage` namespace, the `scipy.ndimage.measurements` namespace is deprecated.
  from scipy.ndimage.measurements import center_of_mass, label
/var/folders/30/gtfddbxx5z3f_hc8v4br4pkh0000gn/T/ipykernel_74912/1514587937.py:10: DeprecationWarning: Please use `label` from the `scipy.ndimage` namespace, the `scipy.ndimage.measurements` namespace is deprecated.
  from scipy.ndimage.measurements import center_of_mass, label


In [24]:
def lagrangian(path,constant,data,var,thr,fp,ini,end,pcen):
    #Start label file so that we do not fuck up the initial one
    obj = xr.DataArray(data*constant,coords=data.coords,attrs=data.attrs,name=var)
    ### Starting with distances and so on
    all_obj = np.where(obj>thr,1,0)
    distance = ndi.distance_transform_edt(all_obj)
    ### This next line is very important, is the one that selects the location of the centers
    ### Tests should be made to found the best centers depending on the correct objects size
    peaks = peak_local_max(np.array(distance),  footprint = np.ones((fp,fp,fp)), labels = np.array(all_obj))
    ### The next commented lines are for python 3.6 and refine the temporal tracking
    ### The script can be use with precision without these following lines
    #is_peak = peak_local_max(np.array(distance), indices=False, footprint=np.ones((1, 1, 1)))
    #tmp_labels = label(is_peak)[0]
    #merged_peaks = center_of_mass(is_peak, tmp_labels, range(1, np.max(tmp_labels)+1))
    coords = np.array(peaks).astype(int)
    mask = np.zeros(distance.shape, dtype=bool)
    mask[tuple(coords.T)] = True
    markers, _ = ndi.label(mask)
    labels = watershed(-distance,markers,mask=all_obj,watershed_line=True)
    #pdb.set_trace()
    if len(np.shape(labels)) == 4:
        obj['Objects'] = (['valid_time','latitude','longitude'],labels[:,0,:,:])
    else:
        obj['Objects'] = (['valid_time','latitude','longitude'],labels)
    obj.to_netcdf(path+'labels_'+ini+'-'+end+'_'+var+'_'+str(pcen)+'_'+str(fp)+'_India.nc')
    return(obj)

In [25]:
### Select the period to study
ini = '2013-01-01'
end = '2023-12-31'
### Indicate the names of the variables you are interesting in and also the name of the files
varis1 = ['pm2p5','pm10','co','tcco']
files = ['particulate_matter_2.5um_India.nc', 'particulate_matter_10um_India.nc',
         'carbon_monoxide_India.nc', 'total_column_carbon_monoxide_India.nc']

### Indicate the path of the raw data and also the path in which you want to save the data
path_obj = '/Volumes/CasiSSD/Postdoc_MSCA-Bridge/Casallas_Loka_project/Data_objects/'
path_data = '/Volumes/CasiSSD/Postdoc_MSCA-Bridge/Casallas_Loka_project/Data_pollutants/'

### In CAMS the units are not in ug_m3 so you need to convert it
### In the case of O3 and SO2 an extra step is needed, and is describe here: 
### https://confluence.ecmwf.int/pages/viewpage.action?pageId=153391710, then the constant is ready to be use
constant = 1e9 
### Directory to save the output
data = {}
### This needs to be change depending the need of the user. 
### The smaller the footprint is, the smaller the objects, so more objects are expected. 
### A randomized search is normally needed to identify the best footprint
footprint = 10
### The percentiles are the thresholds!
percens = [97,97,97,97]
for i,var in enumerate(varis1):
    print('Starting with: '+var)
    ### The try is to avoid creating the objects every time!
    try:
        filename = path_obj+'labels_'+ini+'-'+end+'_'+var+'_'+str(percens[i])+'_'+str(footprint)+'_India.nc'
        obj = xr.open_dataset(filename)
        print('File ready!')
        data[var] = obj
    except:
        print('File not ready, so lets create it')
        data = xr.open_dataset(path_data+files[i])
        data = data[var].loc[ini:end]
        if len(np.shape(data)) == 4:
            data = data[:,0,:,:]
        thr = np.percentile(data,percens[i])*constant
        obj = lagrangian(path_obj,constant,data,var,thr,footprint,ini,end,percens[i])
        data[var] = obj

Starting with: pm2p5
File ready!
Starting with: pm10
File ready!
Starting with: co
File not ready, so lets create it
Starting with: tcco
File ready!
File not ready, so lets create it


PermissionError: [Errno 13] Permission denied: '/Volumes/CasiSSD/Postdoc_MSCA-Bridge/Casallas_Loka_project/Data_objects/labels_2013-01-01-2023-12-31_tcco_97_10_India.nc'